# 3. Gradient Descent Variants

We continue with the network from last week and implement the three variants of gradient descent from the lecture: stochastic, batch and mini-batch GD.

## Setup
The following cells are the solution of last week's notebook. Just run them.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import pandas as pd

X, y = fetch_openml('mnist_784', version=1, return_X_y=True, data_home="./data", cache=True, parser='auto')
y_categorical = pd.get_dummies(y).astype('float32').values
X_scaled = (X/255).astype('float32').values
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_categorical, test_size=0.15, random_state=42)

In [ ]:
import time
import numpy as np

class DeepNeuralNetwork():
    def __init__(self):
        
        # initialize weights randomly
        np.random.seed(0)
        self.w1 = np.random.randn(128, 784)
        self.w2 = np.random.randn(64, 128)
        self.w3 = np.random.randn(10, 64)

    def forward_pass(self, x_train):
        z1 = np.dot(self.w1, x_train)
        a1 = 1/(1 + np.exp(-z1)) # sigmoid activation
        z2 = np.dot(self.w2, a1)
        a2 = 1/(1 + np.exp(-z2)) # sigmoid activation
        z3 = np.dot(self.w3, a2)
        exps = np.exp(z3) 
        a3 = exps / exps.sum() # softmax activation
        
        # we need to remember all values for the gradient calculation
        self.fwdpass = [x_train, z1, a1, z2, a2, z3, a3]
        
        return a3
    
    
    def get_gradients(self, y, y_hat):
        # restore values from foward pass
        a0, z1, a1, z2, a2, z3, a3 = self.fwdpass
        
        # Calculate W3 update
        # for softmax + cross-entropy loss, the error at the output simplifies to y_hat - y
        error = y_hat - y
        gradient_w3 = np.outer(error, a2)

        # Calculate W2 update
        sigmoid_derivative = (np.exp(-z2))/((np.exp(-z2)+1)**2)
        error = np.dot(self.w3.T, error) * sigmoid_derivative
        gradient_w2 = np.outer(error, a1)

        # Calculate W1 update
        sigmoid_derivative = (np.exp(-z1))/((np.exp(-z1)+1)**2)
        error = np.dot(self.w2.T, error) * sigmoid_derivative
        gradient_w1 = np.outer(error, a0)

        return [gradient_w1, gradient_w2, gradient_w3]

In [ ]:
def evaluate(dnn, X, y):
    predictions = []
    for x, y_true in zip(X, y):
        output = dnn.forward_pass(x)
        pred = np.argmax(output) # returns the index of the neuron with the highest value
        predictions.append(pred == np.argmax(y_true))
    return np.mean(predictions)

dnn = DeepNeuralNetwork()
print("Accuracy of the untrained network:", evaluate(dnn, X_test, y_test))

This is the training loop from last week (takes about 1 minute):

In [ ]:
dnn = DeepNeuralNetwork()

no_epochs = 3
learning_rate = 0.01

start_time = time.time()
losses = []
for iteration in range(no_epochs):
    loss = 0
    for x,y in zip(X_train, y_train):
        y_hat = dnn.forward_pass(x)
        gradients = dnn.get_gradients(y, y_hat)
        
        loss += -np.log(np.dot(y, y_hat))
        
        # one step stochastic gradient descent
        dnn.w1 -= learning_rate * gradients[0]
        dnn.w2 -= learning_rate * gradients[1]
        dnn.w3 -= learning_rate * gradients[2]

    loss /= len(X_train) # average loss per sample
    print(f'Epoch: {iteration+1}, Time: {time.time() - start_time:.1f}s, Loss: {loss:.4f}')
    losses.append(loss)

## Task 1: Stochastic gradient descent
The loop above does one update per data point, but always visits the data points in the same order. Change it to real SGD: visit the data points in a **random order** in every epoch.

In [ ]:
dnn = DeepNeuralNetwork()

no_epochs = 3
learning_rate = 0.01

start_time = time.time()
losses_sgd = []
# your code goes here: the training loop from above, but visit the data points in a
# random order in every epoch (hint: np.random.permutation). Save the losses in losses_sgd.


## Task 2: Batch gradient descent
Do **one** weight update per epoch, using the mean gradient over **all** training data points.

*Check:* the losses should be 8.84, 5.67 and 4.87.

In [ ]:
dnn = DeepNeuralNetwork()

no_epochs = 3
learning_rate = 1.0

start_time = time.time()
losses_batch = []
# your code goes here: in every epoch, sum up the gradients of ALL data points and do ONE
# weight update with the mean gradient. Save the losses in losses_batch.


## Task 3: Mini-batch gradient descent
Visit the data points in a random order and do one weight update per mini-batch of 64 data points, using the mean gradient of the mini-batch.

In [ ]:
dnn = DeepNeuralNetwork()

no_epochs = 3
learning_rate = 0.5
batch_size = 64

start_time = time.time()
losses_mini = []
# your code goes here: in every epoch, visit the data points in a random order, split them into
# mini-batches of size batch_size and do one weight update per mini-batch with its mean gradient.
# Save the losses in losses_mini.


## Task 4: Compare

In [ ]:
import matplotlib.pyplot as plt
for losses, name in [(losses_sgd, 'SGD'), (losses_batch, 'Batch GD'), (losses_mini, 'Mini-batch GD (64)')]:
    plt.plot(range(1, len(losses) + 1), losses, marker='o', label=name)
plt.xlabel('number of epochs')
plt.ylabel('average cross-entropy loss')
plt.legend()
plt.show()

1. All three variants compute the same number of gradients per epoch (one per data point). Why is batch GD so much worse after 3 epochs?
2. Compare the time per epoch of the three variants. Why is it so similar?
3. **Bonus:** Run mini-batch GD with the learning rate of SGD (`learning_rate=0.01`). What happens? Why can mini-batch GD use a larger learning rate than SGD?